# Compute summary statistic for the Aspergillus pan-GEM and the respective strain-specific models

## Load libraries

In [ ]:
import medusa
import cobra
import pandas as pd
import numpy as np
from pathlib import Path
from medusa.flux_analysis import flux_balance
from pickle import load
import pickle
from collections import Counter
import re
from itertools import chain

from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

## Load and wrangle models

Load the list of GEMs for all strains generated in *constructEnsemble.ipynb* (gemList_v2). In addition, load the models for the 8 specific strains that were curated further in the script *simulations.ipynb* (gemList_v3_short). Then, construct a fully updated list of all models (gemList_v3_full).

In [ ]:
with open("../data/intermediate/strain-GEMs_automated_187.pickle", 'rb') as infile:
    gemList_v2 = load(infile)

with open("../data/intermediate/strain-GEMs_validated_gapfilled.pickle", 'rb') as infile:
    gemList_v3_short = load(infile)

Set parameter Username
Set parameter LicenseID to value 2681871
Academic license - for non-commercial use only - expires 2026-06-24
Read LP format model from file C:\Users\gilis\AppData\Local\Temp\tmp0wxv60e9.lp
Reading time = 0.02 seconds
: 1760 rows, 2822 columns, 10028 nonzeros
Read LP format model from file C:\Users\gilis\AppData\Local\Temp\tmpmb0jjiaw.lp
Reading time = 0.04 seconds
: 1760 rows, 3892 columns, 15016 nonzeros
Read LP format model from file C:\Users\gilis\AppData\Local\Temp\tmp1bonxt8_.lp
Reading time = 0.05 seconds
: 1760 rows, 3142 columns, 11522 nonzeros
Read LP format model from file C:\Users\gilis\AppData\Local\Temp\tmp86ev25g0.lp
Reading time = 0.05 seconds
: 1760 rows, 3244 columns, 11984 nonzeros
Read LP format model from file C:\Users\gilis\AppData\Local\Temp\tmp6rf2rbeg.lp
Reading time = 0.03 seconds
: 1760 rows, 3304 columns, 12342 nonzeros
Read LP format model from file C:\Users\gilis\AppData\Local\Temp\tmpvn2gjrzq.lp
Reading time = 0.04 seconds
: 1760 row

In [ ]:
# Update the 8 strain-specific models in the list of all models
longList = [mem.id for mem in gemList_v2]
gemList_v3_full = gemList_v2
for mem in gemList_v3_short:
    gemList_v3_full[longList.index(mem.id)] = mem

path = "../model/"
pickle.dump(gemList_v3_full, open(path + "strain-GEMs_187_gapfill-substituted.pickle","wb"))

# For computing the summary statistics, remove the template model constructed in 2008 by Vongsangnak et al. (2008) 
# and the pan-GEM from the list, as these are not actual strain-specific models.
Pan_oryzae = gemList_v3_full.pop(188)
template = gemList_v3_full.pop(0)

# Get Medusa ensemble for easy manipulation
from medusa.core import Ensemble
fullEnsemble = Ensemble(list_of_models = gemList_v3_full,
                        identifier = "testID",
                        name = "testName")

path = "../model/"
pickle.dump(fullEnsemble, open(path+"ensemble_187_gapfill-substituted.pickle","wb"))

Read LP format model from file C:\Users\gilis\AppData\Local\Temp\tmp6dvazvbo.lp
Reading time = 0.05 seconds
: 1760 rows, 3142 columns, 11522 nonzeros


# Compute summary statistics on metabolites, reactions, and genes

## Metabolites

In [ ]:
hlp = [[] for _ in range(len(gemList_v3_full))]
for i in range(len(gemList_v3_full)):
    for mid in gemList_v3_full[i].metabolites:
        hlp[i].append(mid.id)

flattened = [item for sublist in hlp for item in sublist]

print(f"Number of metabolites, counting same metabolite in different compartments separately: {len(set(flattened))}")
print(f"Number of metabolites, counting same metabolite in different compartments as one: {len(set([s.split('[')[0] for s in flattened]))}")

Number of metabolites, counting same metabolite in different compartments separately: 1824
Number of metabolites, counting same metabolite in different compartments as one: 1312


## Reactions

In [6]:
hlp = [[] for _ in range(len(gemList_v3_full))]
for i in range(len(gemList_v3_full)):
    for rid in gemList_v3_full[i].reactions:
        hlp[i].append(rid.id)

# must first remove all reaction IDs
flattened = [item for sublist in hlp for item in sublist]
# Count the frequencies
count = Counter(flattened)

# Categorize strings by frequency
more_than_178 = sum(1 for value in count.values() if value >= 178)
between_2_and_178 = sum(1 for value in count.values() if 2 <= value < 178)
exactly_1 = sum(1 for value in count.values() if value <= 1)

# Print the results
print(f"Reactions present in more than 178 strains: {more_than_178}")
print(f"Reactions present in between 2 and 178 strains: {between_2_and_178}")
print(f"Reactions present in a single strain: {exactly_1}")
print(f"Total number of reactions: {len(count)}")

Reactions present in more than 178 strains: 1502
Reactions present in between 2 and 178 strains: 433
Reactions present in a single strain: 90
Total number of reactions: 2025


## Genes

In [7]:
from collections import Counter
hlp = [[] for _ in range(len(gemList_v3_full))]
for i in range(len(gemList_v3_full)): # do not take entry 157, this is the pan model
    for gid in gemList_v3_full[i].genes:
        #if rid.id != "Ani_r163a":
        hlp[i].append(gid.id)

# must first remove all reaction IDs

flattened = [item for sublist in hlp for item in sublist]
# Count the frequencies
count = Counter(flattened)

# Categorize strings by frequency
more_than_178 = sum(1 for value in count.values() if value >= 178)
between_2_and_178 = sum(1 for value in count.values() if 2 <= value < 178)
exactly_1 = sum(1 for value in count.values() if value <= 1)

# Print the results
print(f"Genes present in more than 178 strains: {more_than_178}")
print(f"Genes present in between 2 and 178 strains: {between_2_and_178}")
print(f"Genes present in a single strain: {exactly_1}")
print(f"Total number of genes: {len(count)}")

Genes present in more than 178 strains: 1152
Genes present in between 2 and 178 strains: 187
Genes present in a single strain: 58
Total number of genes: 1397


In [8]:
len(template.reactions)
len(Pan_oryzae.reactions)

1392

1918

# Compute summary statistics on the orthologous gene clustering

In [9]:
BPGA = pd.read_csv('../data/genome/BPGA2ortho_GEM_custom_187_50.csv', sep=';', low_memory=False)

# Replace empty strings with NaN
BPGA.replace("", np.nan, inplace=True)

# Compute number of genomes where present (non-NA entries from column 7 onwards)
BPGA["present_in_n_genomes"] = BPGA.iloc[:, 6:].notna().sum(axis=1)

# Filter rows where present_in_n_genomes != 0
BPGA_sub = BPGA[BPGA["present_in_n_genomes"] != 0].copy()

print(f"Total number of orthologous gene cluster accross all strains: {len(BPGA_sub)}")

Total number of orthologous gene cluster accross all strains: 11949


In [10]:
def classify_cluster(n):
    if 178 <= n <= 187:
        return "core"
    elif 2 <= n < 178:
        return "accessory"
    elif n == 1:
        return "singleton"
    else:
        return np.nan

BPGA_sub["cluster_type_manual"] = BPGA_sub["present_in_n_genomes"].apply(classify_cluster)

# Frequency table (like R's table())
print(BPGA_sub["cluster_type_manual"].value_counts())

cluster_type_manual
core         8841
accessory    2298
singleton     810
Name: count, dtype: int64


# Summary statistics on the reaction pathways

In [11]:
unique_reaction_notes = {}

for gem in gemList_v3_full:
    for rxn in gem.reactions:
        note = rxn.notes.get("NOTES")
        note = "Added from oryzae" if not note or note == {} else note
        note = "Added from KEGG via addRxnsAndMets()" if note == 'Added from OtherAsp via addRxnsAndMets()' else note

        # Add only if this reaction hasn't been recorded yet
        if rxn.id not in unique_reaction_notes:
            unique_reaction_notes[rxn.id] = note

for rxn_id, note in unique_reaction_notes.items():
    if rxn_id.lower().startswith("gap"):  # lowercase ensures "Gap" or "gap"
        unique_reaction_notes[rxn_id] = "Added by gapfilling"

note_counts = Counter(unique_reaction_notes.values())
note_counts

Counter({'Added from oryzae': 1426,
         'Added from KEGG via addRxnsAndMets()': 454,
         'Added by gapfilling': 59,
         'Added from niger via addRxnsAndMets()': 52,
         'Added from fumigatus via addRxnsAndMets()': 34})

In [12]:
unique_reactions = {}
for gem in gemList_v3_full:
    for rxn in gem.reactions:
        unique_reactions[rxn.id] = rxn   # overwrites duplicates automatically

len(unique_reactions)

2025

In [13]:
# Transport reactions
transportNames = pd.read_csv(
    "../data/intermediate/transportRxns.csv",
    sep=";",
    header=None
)[0].tolist()
reaction_ids = unique_reactions.keys()
transportIdx = [
    i for i, rid in enumerate(reaction_ids)
    if rid in transportNames
]

# Number of transport reactions
num_transport_rxns = len(transportIdx)
print(num_transport_rxns)

# transport_rxns = [unique_reactions[i] for i in transportIdx]
values = list(unique_reactions.values())
transport_rxns = [values[i] for i in transportIdx]

num_with_gene_assoc = sum(
    rxn.gene_reaction_rule not in ("", None)
    for rxn in transport_rxns
)
print(num_with_gene_assoc)

num_without_gene_assoc = sum(
    rxn.gene_reaction_rule in ("", None)
    for rxn in transport_rxns
)
print(num_without_gene_assoc)

keys = list(unique_reactions.keys())
for i in sorted(transportIdx, reverse=True):
    del unique_reactions[keys[i]]

211
66
145


In [14]:
# Regex for exchange reactions
exchange_pattern = re.compile(r"^\s*(<=>|--> |<--)|(<=>|--> |<--)\s*$")

# Indices of exchange reactions
exchange_idx = [
    i for i, rxn in enumerate(unique_reactions.values())
    if exchange_pattern.search(rxn.reaction)
]

print(len(exchange_idx))

keys = list(unique_reactions.keys())
for i in sorted(exchange_idx, reverse=True):
    del unique_reactions[keys[i]]

162


In [ ]:
values = list(unique_reactions.values())

# Extract bracketed parts from each reaction
brackets = [re.findall(r'(?<=\[)[^\]]+', rxn.reaction) for rxn in values]

# Count unique entries per reaction
unique_counts = [len(set(b)) for b in brackets]

# Identify reactions with exactly one unique bracket
rows_with_one_unique = [i for i, count in enumerate(unique_counts) if count == 1]
print("Number of reactions with 1 unique bracket:", len(rows_with_one_unique))

Number of reactions with 1 unique bracket: 1627


In [ ]:
# Table of unique values in reactions with one unique bracket
one_unique_values = list(chain.from_iterable([list(set(brackets[i])) for i in rows_with_one_unique]))
print("Counts of unique values in these reactions:")
print(Counter(one_unique_values))

Counts of unique values in these reactions:
Counter({'c': 1244, 'm': 220, 'e': 83, 'p': 80})


In [ ]:
# Table of counts of unique brackets across all reactions
print("Distribution of unique bracket counts across all reactions:")
print(Counter(unique_counts))  # or pd.Series(unique_counts).value_counts().sort_index()

Distribution of unique bracket counts across all reactions:
Counter({1: 1627, 2: 25})


In [ ]:
# 6Subset reactions with more than one unique bracket
reactions_with_multiple = [rxn for rxn, count in zip(values, unique_counts) if count > 1]

# save subset to Excel
df = pd.DataFrame({
    "reaction": [rxn.reaction for rxn in reactions_with_multiple],
    "id": [rxn.id for rxn in reactions_with_multiple]  # adjust / add more attributes if needed
})

df.to_excel("../data/intermediate/ambiguous_reactions.xlsx", index=False)

# Comparison of strain-specific models

Currently, these results are not discussed in the manuscript. There, we only compare the models in the tSNE clustering based on reaction identifier content.

In [ ]:
# import matplotlib.pyplot as plt
# import itertools
# import pandas as pd

# gemList = gemList_v3_full

# nr_reactions = [len(gem.reactions) for gem in gemList]
# nr_genes = [len(gem.genes) for gem in gemList]

# plt.hist(nr_reactions, bins=40, edgecolor='black')  # You can adjust `bins` as needed
# plt.xlabel('Number of Reactions')
# plt.ylabel('Frequency')
# plt.show()

# plt.hist(nr_genes, bins=40, edgecolor='black')  # You can adjust `bins` as needed
# plt.xlabel('Number of Genes')
# plt.ylabel('Frequency')
# plt.show()

# submembers = [mem.id for mem in gemList_v3]

# gemList_target = []
# for i in range(len(gemList)):
#     if(gemList[i].id in submembers):
#         gemList_target.append(gemList[i].copy())

# [gem.id for gem in gemList_target]
# [len(gem.reactions) for gem in gemList_target]
# [len(gem.genes) for gem in gemList_target]

# result = []
# # subset_models = [gemList_target[i] for i in [1,2,3,4,5,6,8]] # only focus on the industrial strains for now
# subset_models = gemList_target

# # Iterate over all unique pairs of models
# for model1, model2 in itertools.combinations(subset_models, 2):

#     # Genes of model1 and model2
#     genes1 = set([gene.id for gene in model1.genes])
#     genes2 = set([gene.id for gene in model2.genes])
#     overlapGenes = len(genes1 & genes2)  # Intersection of gene sets

#     # Reactions of model1 and model2
#     reactions1 = set([rxn.id for rxn in model1.reactions])
#     reactions2 = set([rxn.id for rxn in model2.reactions])
#     overlapRxns = len(reactions1 & reactions2)  # Intersection of reaction sets
    
#     # Append the result to the data list
#     result.append([model1.id, model2.id, len(genes1), len(genes2), overlapGenes, len(reactions1), len(reactions2), overlapRxns])

# # Create a DataFrame from the results
# df = pd.DataFrame(result, columns=['Model 1', 'Model 2', 'Genes in Model 1', 'Genes in Model 2', 'Overlap genes', 'Reactions in Model 1', 'Reactions in Model 2', 'Overlap reactions'])

# # Show the DataFrame
# df

# # How many genes/reactions shared across all industrial strains??
# shared_reactions = set([rxn.id for rxn in subset_models[0].reactions])
# for model in subset_models[1:]:
#     shared_reactions &= set([rxn.id for rxn in model.reactions])
# len(shared_reactions)
# [len(gem.reactions) for gem in subset_models]

# shared_genes = set([gene.id for gene in subset_models[0].genes])
# for model in subset_models[1:]:
#     shared_genes &= set([gene.id for gene in model.genes])
# len(shared_genes)
# [len(gem.genes) for gem in subset_models]
# # Start with an empty set for all reactions
# all_reactions = set()

# # Add reactions from all models to the all_reactions set
# for model in subset_models:
#     all_reactions.update(set([rxn.id for rxn in model.reactions]))

# # Start with the reactions of the first model as the initial set
# shared_reactions = set([rxn.id for rxn in subset_models[0].reactions])

# # Intersect with the reactions of each subsequent model
# for model in subset_models[1:]:
#     shared_reactions &= set([rxn.id for rxn in model.reactions])

# # Reactions that are not shared across all models
# non_shared_reactions = all_reactions - shared_reactions

# len(non_shared_reactions) # it is all about 258 reactions